# Comparing Performance of TGradAMI vs. TGrad (for EVI and Rainfall data).


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Import from local directory
import sys

sys.path.insert(0, '../src')
from TemporalGP.TGP.tgrad_ami import TGradAMI
from TemporalGP.TGP.t_graank import TGrad

Configure TGradAMI algorithm hyperparameters.

In [ ]:
# Hyperparameters

# f_path = "../datasets/air_quality1k.csv"
# f_path = "../datasets/air_quality25.csv"

f_path_evi = "../datasets/ke_evi_data_2k.csv"
f_path_rain = "../datasets/ke_rain_data_2k.csv"

eq = False
min_sup = 0.1
tgt_col = 1
tgt_cols = [1, 2, 3, 4]
min_rep = 0.75
mi_err_margin = 0.0001
eval_mode = True
clustering_method = False

Visualize the top 5 rows of the dataset.

In [ ]:
rain_data = pd.read_csv(f_path_rain)
evi_data = pd.read_csv(f_path_evi)


print(rain_data.head())
print("\n")
print(evi_data.head())

Initialize TGrad and TGradAMI algorithms by creating objects for different target columns.

In [ ]:
rain_tgrads_ami = []
evi_tgrads_ami = []
for t_col in tgt_cols:
    t_grad_rain = TGradAMI(f_path_rain, min_sup, eq, target_col=t_col, min_rep=min_rep, min_error=mi_err_margin)
    t_grad_evi = TGradAMI(f_path_evi, min_sup, eq, target_col=t_col, min_rep=min_rep, min_error=mi_err_margin)

    rain_tgrads_ami.append(t_grad_rain)
    evi_tgrads_ami.append(t_grad_evi)


rain_tgrads = []
evi_tgrads = []
for t_col in tgt_cols:
    t_grad_rain = TGrad(f_path_rain, min_sup, eq, target_col=t_col, min_rep=min_rep)
    t_grad_evi = TGrad(f_path_evi, min_sup, eq, target_col=t_col, min_rep=min_rep)

    rain_tgrads.append(t_grad_rain)
    evi_tgrads.append(t_grad_evi)

Run the algorithm for mining FTGPs (Fuzzy Temporal Gradual Patterns) in evaluation mode. The algorithm returns a number of results in a dict format.

In [ ]:
# rain_tgrad_data = []
# evi_tgrad_data = []
for i, t_grad_rain in enumerate(rain_tgrads_ami):
    eval_dict_rain = t_grad_rain.discover_tgp(use_clustering=clustering_method, eval_mode=eval_mode)
    transform_steps = eval_dict_rain['Transformation Steps']
    eval_dict_evi = evi_tgrads_ami[i].discover_tgp(use_clustering=clustering_method, transformation_steps=transform_steps, eval_mode=eval_mode)

    # rain_tgrad_data.append(eval_dict_rain)
    # evi_tgrad_data.append(eval_dict_evi)

In [ ]:
for i in range(len(rain_tgrads)):
    rain_tgrads[i].discover_tgp()
    evi_tgrads[i].discover_tgp()


* Tabulate the mined FTGPs with their corresponding parameters

## TGradAMI FTGPs

In [ ]:
patterns = [t_grad_rain.display_patterns_as_df for t_grad_rain in rain_tgrads_ami]  # Collect all patterns into a list
res_rain_df = pd.concat(patterns, ignore_index=True)  # Combine them in one go
res_rain_df

In [ ]:
patterns = [t_grad_evi.display_patterns_as_df for t_grad_evi in evi_tgrads_ami]  # Collect all patterns into a list
res_evi_df = pd.concat(patterns, ignore_index=True)  # Combine them in one go
res_evi_df

## TGrad FTGPs

In [ ]:
patterns = [t_grad_rain.display_patterns_as_df for t_grad_rain in rain_tgrads]
res_rain_df2 = pd.concat(patterns, ignore_index=True)
res_rain_df2

In [ ]:
patterns = [t_grad_evi.display_patterns_as_df for t_grad_evi in evi_tgrads]
res_evi_df2 = pd.concat(patterns, ignore_index=True)
res_evi_df2

## Compute recall, precision and F1 score of extracted FTGPs

In [ ]:
tgrad_ami_lst_pat_rain = []
for tgrad in rain_tgrads_ami:
    tgrad_ami_lst_pat_rain.extend(tgrad.gradual_patterns)

tgrad_ami_lst_pat_evi = []
for tgrad in evi_tgrads_ami:
    tgrad_ami_lst_pat_evi.extend(tgrad.gradual_patterns)

tgrad_lst_pat_rain = []
for tgrad in rain_tgrads:
    tgrad_lst_pat_rain.extend(tgrad.gradual_patterns)

tgrad_lst_pat_evi = []
for tgrad in evi_tgrads:
    tgrad_lst_pat_evi.extend(tgrad.gradual_patterns)

* True Positive (TP): an FTGP that appears in both **Rain** and **EVI** datasets with support >= 0.5
* False Positive (FP): an FTGP that appears in **Rain** dataset with support >= 0.5 but appears in **EVI** dataset with support < 0.5
* False Negative (FN): an FTGP that appears in **Rain** dataset with support < 0.5 but appears in **EVI** dataset with support >= 0.5
* True Negative (TN): an FTGP that appears in both **Rain** and **EVI** datasets with support < 0.5

In [ ]:
res_tgrad_ami = {"TP": 0, "FP": 0, "FN": 0, "TN": 0, "Missing": 0}
res_tgrad = {"TP": 0, "FP": 0, "FN": 0, "TN": 0, "Missing": 0}

def exists_in(pat, lst):
    for pat_i in lst:
        if pat.is_similar_to(pat_i):
            return True
    return False

total = 0
already_seen = set()
for pat1 in tgrad_ami_lst_pat_rain:
    for pat2 in tgrad_ami_lst_pat_evi:
        if pat1.is_similar_to(pat2) and not exists_in(pat2, already_seen):
            already_seen.add(pat2)
            if pat1.support >= 0.5 and pat2.support >= 0.5:
                res_tgrad_ami["TP"] += 1
                total += 1
            elif pat1.support >= 0.5 > pat2.support:
                res_tgrad_ami["FP"] += 1
                total += 1
            elif pat1.support < 0.5 <= pat2.support:
                res_tgrad_ami["FN"] += 1
                total += 1
            elif pat1.support < 0.5 > pat2.support:
                res_tgrad_ami["TN"] += 1
                total += 1
res_tgrad_ami["Missing"] = len(tgrad_ami_lst_pat_rain) - total


total = 0
already_seen = set()
for pat1 in tgrad_lst_pat_rain:
    for pat2 in tgrad_lst_pat_evi:
        if pat1.is_similar_to(pat2) and pat2 not in already_seen:
            already_seen.add(pat2)
            if pat1.support >= 0.5 and pat2.support >= 0.5:
                res_tgrad["TP"] += 1
                total += 1
            elif pat1.support >= 0.5 > pat2.support:
                res_tgrad["FP"] += 1
                total += 1
            elif pat1.support < 0.5 <= pat2.support:
                res_tgrad["FN"] += 1
                total += 1
            elif pat1.support < 0.5 > pat2.support:
                res_tgrad["TN"] += 1
                total += 1
res_tgrad["Missing"] = len(tgrad_lst_pat_rain) - total

print(f"TGradAMI: {res_tgrad_ami}")
print(f"TGrad: {res_tgrad}")

In [ ]:
cat_tgrad_ami = list(res_tgrad_ami.keys())
cat_tgrad = list(res_tgrad_ami.keys())
vals_tgrad_ami = list(res_tgrad.values())
vals_tgrad = list(res_tgrad.values())

# Create plot
plt.figure(figsize=(8, 5))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
colors = ['#4CAF50', '#F44336', '#FF9800', '#2196F3'] # Green, Red, Orange, Blue
ax1.bar(cat_tgrad_ami, vals_tgrad_ami, color=colors)
ax2.bar(cat_tgrad, vals_tgrad, color=colors)

# Add styling
ax1.set_title('Confusion Matrix (TGradAMI)')
ax1.set_xlabel('Metric Type')
ax1.set_ylabel('Count')
ax2.set_title('Confusion Matrix (TGrad)')
ax2.set_xlabel('Metric Type')
ax2.set_ylabel('Count')

# Add value labels on top of bars
for i, v in enumerate(vals_tgrad_ami):
    ax1.text(i, v + 0.5, str(v), ha='center', fontweight='bold')

for i, v in enumerate(vals_tgrad):
    ax2.text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.show()

In [ ]:
def compute_performance(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return round(precision, 3), round(recall,3), round(f1,3)

In [ ]:
tgrad_ami_pre, tgrad_ami_rec, tgrad_ami_f1 = compute_performance(res_tgrad_ami["TP"], res_tgrad_ami["FP"], res_tgrad_ami["FN"])
tgrad_pre, tgrad_rec, tgrad_f1 = compute_performance(res_tgrad["TP"], res_tgrad["FP"], res_tgrad["FN"])

print(f"TGradAMI\nPrecision: {tgrad_ami_pre}, Recall: {tgrad_ami_rec}, F1: {tgrad_ami_f1}\n")
print(f"TGrad\nPrecision: {tgrad_pre}, Recall: {tgrad_rec}, F1: {tgrad_f1}")